# CKD Pipeline Runner — Revision 1 (MLWA-D-26-01812) — Google Colab

## 1. Environment (pinned to the configuration that reproduced the paper)

In [ ]:
# Versions of the Colab session that reproduced every published value
# (2026-11-13: Linux, 2 cores, XGBoost 3.4.1 Linux wheel built with GCC 10.3.1).
# First run: installs any mismatched package; if a package that Colab had
# already loaded is changed, the runtime restarts automatically.
# After a restart: Runtime -> Run all.
import importlib.metadata as md
import os
import subprocess
import sys

PINNED = {
    "numpy": "2.4.6",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
    "xgboost": "3.4.1",
    "lightgbm": "4.7.0",
    "imbalanced-learn": "0.14.2",
    "statsmodels": "0.15.0",
    "joblib": "1.6.0",
    "threadpoolctl": "3.7.0",
}
IMPORT_NAMES = {"scikit-learn": "sklearn", "imbalanced-learn": "imblearn"}

def installed(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

mismatch = {p: installed(p) for p, v in PINNED.items() if installed(p) != v}

if mismatch:
    print("Installing pinned versions for:", mismatch)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           *[f"{p}=={PINNED[p]}" for p in mismatch]])
    already_loaded = [p for p in mismatch
                      if IMPORT_NAMES.get(p, p.replace("-", "_")) in sys.modules]
    if already_loaded:
        print("Restarting runtime (already-loaded packages changed:", already_loaded, ")")
        print("When it reconnects: Runtime -> Run all.")
        os.kill(os.getpid(), 9)

for p, v in PINNED.items():
    got = installed(p)
    print(f"{p:<18} {got:<10} {'OK' if got == v else 'MISMATCH'}")
if any(installed(p) != v for p, v in PINNED.items()):
    raise RuntimeError("Pinned versions not active. Restart the runtime and Run all again.")

import platform
import xgboost
b = xgboost.build_info()
print("\nPython      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("CPU cores   :", os.cpu_count())
print("XGBoost build: GCC", ".".join(map(str, b.get("GCC_VERSION", []))),
      "(reproduction used GCC 10.3.1)")

## 2. Google Drive, project folder, environment record

In [ ]:
import json
import shutil
import time
import datetime
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_FOLDER = Path("/content/drive/MyDrive/CKD-Paper-Final")
ZIP_PATH = DRIVE_FOLDER / "ckd-project.zip"

EXPECTED_MODULES = 42   # modules 00-41 (40 and 41 are the figure stages)

def module_count(folder):
    return len(list((folder / "modules").glob("[0-9][0-9]_*.py")))

def find_project(base):
    """Every folder (up to 3 levels deep) holding pipeline_runner.py + modules/."""
    hits = []
    for runner_file in base.glob("**/pipeline_runner.py"):
        folder = runner_file.parent
        depth = len(folder.relative_to(base).parts)
        if depth <= 3 and (folder / "modules").is_dir():
            hits.append((module_count(folder), folder))
    return hits

candidates = find_project(DRIVE_FOLDER)
complete = [f for n, f in candidates if n == EXPECTED_MODULES]

if not complete and ZIP_PATH.exists():
    print("No complete project found; extracting", ZIP_PATH.name)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DRIVE_FOLDER)
    candidates = find_project(DRIVE_FOLDER)
    complete = [f for n, f in candidates if n == EXPECTED_MODULES]

for n, f in sorted(candidates, key=lambda t: str(t[1])):
    print(f"found: {f.relative_to(DRIVE_FOLDER)}  ({n} modules){'  <- complete' if n == EXPECTED_MODULES else '  (old/incomplete, ignored)'}")

if not complete:
    raise FileNotFoundError(
        f"No folder with all {EXPECTED_MODULES} modules under {DRIVE_FOLDER}. "
        "Upload ckd-project.zip there and run this cell again."
    )

# Prefer the shallowest complete copy (e.g. ckd-project over ckd-project/ckd-project).
PROJECT_ROOT = min(complete, key=lambda f: len(f.relative_to(DRIVE_FOLDER).parts))
MODULE_NAMES = sorted(p.stem for p in (PROJECT_ROOT / "modules").glob("[0-9][0-9]_*.py"))
print("\nUsing PROJECT_ROOT:", PROJECT_ROOT)

# Keep earlier results: move them to a dated backup instead of deleting.
RESULTS = PROJECT_ROOT / "mlwa_results"
if RESULTS.exists() and any(RESULTS.iterdir()):
    backup = PROJECT_ROOT / f"mlwa_results_backup_{datetime.datetime.now():%Y%m%d_%H%M%S}"
    shutil.move(str(RESULTS), str(backup))
    print("Previous results moved to:", backup.name)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Environment record for this run.
from threadpoolctl import threadpool_info
RESULTS.mkdir(parents=True, exist_ok=True)
env_record = {
    "captured_at": datetime.datetime.now().isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu_model": subprocess.run("lscpu | grep 'Model name'", shell=True,
                                capture_output=True, text=True).stdout.strip(),
    "cpu_count": os.cpu_count(),
    "versions": {p: installed(p) for p in PINNED},
    "xgboost_build": xgboost.build_info(),
    "thread_pools": [{k: t.get(k) for k in ("user_api", "internal_api", "num_threads", "version")}
                     for t in threadpool_info()],
}
(RESULTS / "run_environment.json").write_text(json.dumps(env_record, indent=2, default=str))
(RESULTS / "run_pip_freeze.txt").write_text(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Modules     :", len(MODULE_NAMES))
print("Environment record saved: mlwa_results/run_environment.json, run_pip_freeze.txt")

## 3. Initialize pipeline

In [ ]:
import importlib
import pipeline_runner
importlib.reload(pipeline_runner)

from pipeline_runner import CKDPipelineRunner, banner, finish_block
from IPython.display import display

runner = CKDPipelineRunner(PROJECT_ROOT)

# True  = rebuild nhanes_ckd_risk_clean.csv from CDC/NCHS raw files
#         (corrected missing-creatinine/ACR exclusion; n = 5,063).
# False = use the shipped corrected CSV. Both give identical results.
RUN_DATA_PREPARATION = True

# Stop after Experiment 1 / Experiment 7 if published values do not reproduce.
REQUIRE_EXACT_REPRODUCTION = True

PIPELINE_T0 = time.time()

## 4. Original pipeline (modules 00–25)

In [ ]:
banner("SETUP: Paths, imports, and reproducibility settings")
t0 = time.time()

runner.setup()

finish_block(t0)

In [ ]:
banner("DATA PREPARATION: NHANES 2017–2018")
t0 = time.time()

if RUN_DATA_PREPARATION:
    runner.prepare_data()
else:
    print("Skipped. Using the shipped corrected development CSV (n = 5,063).")

finish_block(t0)

In [ ]:
banner("DEVELOPMENT DATA: 2017–2018 cohort and survey design variables")
t0 = time.time()

runner.load_development_data()

finish_block(t0)

In [ ]:
banner("PREPROCESSING: Imputation, encoding, and scaling")
t0 = time.time()

runner.build_preprocessor()

finish_block(t0)

In [ ]:
banner("MODEL SETUP: Candidate algorithms")
t0 = time.time()

runner.define_candidate_models()

finish_block(t0)

In [ ]:
banner("EVALUATION METRICS")
t0 = time.time()

runner.define_metrics()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 1: Nested five-fold cross-validation")
t0 = time.time()

runner.run_nested_cv()

model_comparison = runner.get("model_comparison")
if model_comparison is not None:
    display(model_comparison.round(4))

finish_block(t0)

In [ ]:
banner("REPRODUCTION CHECK 1: nested CV (Table 2)")
published_cv = {
    "Random Forest":        (0.7879, 0.1297),
    "XGBoost":              (0.7871, 0.1320),
    "HistGradientBoosting": (0.7819, 0.1237),
    "LightGBM":             (0.7817, 0.1572),
    "Logistic Regression":  (0.7808, 0.1364),
}
mc = runner.get("model_comparison").set_index("model")
all_match = True
for model, (auc, brier) in published_cv.items():
    a = round(float(mc.loc[model, "mean_roc_auc"]), 4)
    b = round(float(mc.loc[model, "mean_brier"]), 4)
    ok = (a == auc) and (b == brier)
    all_match &= ok
    print(f"{model:<22} AUC {a:.4f} vs {auc:.4f} | Brier {b:.4f} vs {brier:.4f} | {'MATCH' if ok else 'DIFFERENT'}")
selected = runner.get("SELECTED_MODEL_NAME")
print("\nSelected model:", selected, "(published: HistGradientBoosting)")
all_match &= selected == "HistGradientBoosting"
print("\nREPRODUCTION CHECK 1:", "PASSED" if all_match else "FAILED")
if REQUIRE_EXACT_REPRODUCTION and not all_match:
    raise RuntimeError("Nested CV does not reproduce the published Table 2. Stopping before the long run.")

In [ ]:
banner("EXPERIMENT 2: Bootstrap uncertainty")
t0 = time.time()

runner.run_bootstrap_uncertainty()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 3: Paired outer-fold model comparison")
t0 = time.time()

runner.run_paired_model_comparison()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 4: Final tuning on the full development cohort")
t0 = time.time()

runner.tune_final_model()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 5: Raw vs sigmoid vs isotonic calibration")
t0 = time.time()

runner.calibrate_model()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 6: Freeze the screening threshold")
t0 = time.time()

runner.freeze_threshold()

print("Frozen threshold:", runner.get("SCREENING_THRESHOLD"))

finish_block(t0)

In [ ]:
banner("FINAL MODEL: Fit the frozen calibrated probability model")
t0 = time.time()

runner.fit_final_model()

finish_block(t0)

In [ ]:
banner("TEMPORAL DATA: Build NHANES 2021–2023 validation cohort")
t0 = time.time()

runner.build_temporal_cohort()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 7: Temporal validation")
t0 = time.time()

runner.run_temporal_validation()

finish_block(t0)

In [ ]:
banner("REPRODUCTION CHECK 2: frozen model and temporal validation")
published = {
    "selected model": ("HistGradientBoosting", runner.get("SELECTED_MODEL_NAME")),
    "calibration":    ("Sigmoid", runner.get("BEST_CALIBRATION")),
    "threshold":      (0.1239, round(float(runner.get("SCREENING_THRESHOLD")), 4)),
    "temporal AUC":   (0.7949, round(float(runner.get("temporal_unweighted")["roc_auc"]), 4)),
    "temporal sens":  (0.8251, round(float(runner.get("temporal_unweighted")["sensitivity"]), 4)),
    "temporal spec":  (0.6037, round(float(runner.get("temporal_unweighted")["specificity"]), 4)),
    "weighted sens":  (0.7671, round(float(runner.get("temporal_weighted")["sensitivity"]), 4)),
}
all_match = True
for name, (pub, new_val) in published.items():
    ok = pub == new_val
    all_match &= ok
    print(f"{name:<15} published {pub} | this run {new_val} | {'MATCH' if ok else 'DIFFERENT'}")
print("\nREPRODUCTION CHECK 2:", "PASSED" if all_match else "FAILED")
if REQUIRE_EXACT_REPRODUCTION and not all_match:
    raise RuntimeError("Frozen model does not reproduce the published results. Stopping.")

In [ ]:
banner("EXPERIMENT 8: Temporal calibration transport")
t0 = time.time()

runner.run_temporal_calibration()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 9: Decision-curve analysis")
t0 = time.time()

runner.run_decision_curve()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 10: Age-removal sensitivity")
t0 = time.time()

runner.run_age_sensitivity()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 11: Low-eGFR vs albuminuria components")
t0 = time.time()

runner.run_outcome_component_sensitivity()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 12: Race and physical-activity feature-set sensitivity")
t0 = time.time()

runner.run_feature_set_sensitivity()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 13: Temporal subgroup performance and calibration")
t0 = time.time()

runner.run_subgroup_analysis()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 14: Age- and diabetes-stratified thresholds")
t0 = time.time()

runner.run_stratified_thresholds()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 15: Class weighting, SMOTE, and SMOTEENN")
t0 = time.time()

runner.run_imbalance_sensitivity()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 16: Survey-weighted evaluation")
t0 = time.time()

runner.run_survey_weighted_evaluation()

finish_block(t0)

In [ ]:
banner("EXPERIMENT 17: Temporal permutation importance")
t0 = time.time()

runner.run_permutation_importance()

finish_block(t0)

In [ ]:
banner("FINAL OUTPUTS: Reproducibility record and master tables")
t0 = time.time()

runner.finalize_outputs()

finish_block(t0)

## 5. Reviewer-revision analyses (modules 26–39)

In [ ]:
banner("REVISION SETUP: Helpers, survey design, revision output folder")
t0 = time.time()

runner.revision_setup()

finish_block(t0)

In [ ]:
banner("REVISION A: Prevalence n/N, included vs excluded, outcome components (R2.2, R2.6, R3m2)")
t0 = time.time()

runner.run_cohort_description()

finish_block(t0)

In [ ]:
banner("REVISION B: One-SE rule, calibration methods, calibrated family comparison, fully nested procedure (R1.2, R2.4, R2.5, R3.1, R3m1, R4.5)")
t0 = time.time()

runner.run_model_selection_revision()

finish_block(t0)

In [ ]:
banner("REVISION C: Calibrated LR, reduced predictor sets, risk-factor rules, testing burden (R1, R1.1, R4.1)")
t0 = time.time()

runner.run_comparators_and_ablation()

finish_block(t0)

In [ ]:
banner("REVISION D: Survey-weighted 80% sensitivity threshold (R1.3, R2.7, R3.3, R4.4)")
t0 = time.time()

runner.run_survey_weighted_threshold()

finish_block(t0)

In [ ]:
banner("REVISION E: Survey-design-aware uncertainty, Rao-Wu bootstrap (R3.2)")
t0 = time.time()

runner.run_design_based_uncertainty()

finish_block(t0)

In [ ]:
banner("REVISION F: Calibration CIs, ECE binning, calibration near the threshold (R3.4)")
t0 = time.time()

runner.run_calibration_uncertainty()

finish_block(t0)

In [ ]:
banner("REVISION G: Subgroup CIs and screening policies (R2.8, R2.9, R3m5, R4.2)")
t0 = time.time()

runner.run_subgroup_uncertainty_and_policies()

finish_block(t0)

In [ ]:
banner("REVISION H: Age dependence and outcome components (R2.11, R3m2, R4.3)")
t0 = time.time()

runner.run_age_dependence()

finish_block(t0)

In [ ]:
banner("REVISION I: Alcohol-intake missingness sensitivity (R2.3, R3m4, R4 minor)")
t0 = time.time()

runner.run_alcohol_missingness()

finish_block(t0)

In [ ]:
banner("REVISION J: Screening scenarios and DCA threshold interpretation (R2.10, R3.5)")
t0 = time.time()

runner.run_screening_scenarios()

finish_block(t0)

In [ ]:
banner("REVISION K: Correlation-aware permutation importance (R3m3)")
t0 = time.time()

runner.run_correlated_importance()

finish_block(t0)

In [ ]:
banner("REVISION L: Blood-pressure protocol sensitivity, BPXO_J (R4 minor)")
t0 = time.time()

runner.run_blood_pressure_protocol()

finish_block(t0)

## 6. Final outputs

In [ ]:
banner("REVISION SUMMARY: Comment map, key numbers, reproducibility record")
t0 = time.time()

runner.finalize_revision_outputs()

comment_map = runner.get("comment_output_map")
if comment_map is not None:
    display(comment_map)

finish_block(t0)

## 7. Figures

Module 40 saves the numbers behind each figure to `mlwa_results/figure_data/`; module 41 draws the six manuscript figures and Supplementary Figure S1 into `mlwa_results/figures/` (EPS, PDF, PNG). To restyle a figure later without re-running the pipeline, edit `make_figures.py` in the project folder and run `%run make_figures.py 1` (or any figure number).


In [ ]:
banner("FIGURES A: Save the data behind every figure")
t0 = time.time()

runner.export_figure_data()

finish_block(t0)

In [ ]:
banner("FIGURES B: Draw Figures 1-6 and Supplementary Figure S1")
t0 = time.time()

runner.make_figures()

finish_block(t0)

In [ ]:
banner("CKD PIPELINE COMPLETE (PRIMARY ANALYSIS + REVISION 1)")

print("Project root      :", PROJECT_ROOT)
print("Primary results   :", runner.get("RESULTS_DIR"))
print("Revision results  :", runner.get("REVISION_DIR"))
print("Figures           :", runner.get("RESULTS_DIR") / "figures")
print(f"Total run time    : {(time.time() - PIPELINE_T0) / 60:.1f} minutes")